## <span style="color: #eab308">Table of Contents</span>

- [Current Project Status](#current-project-status)
- [Tech Stack and Architecture](#tech-stack-and-architecture)
- [Environment Variables You Need](#environment-variables-you-need)
- [How the App Starts, Step by Step](#how-the-app-starts-step-by-step)
- [What to Type in the Terminal](#what-to-type-in-the-terminal)
- [Troubleshooting and Notes](#troubleshooting-and-notes)
- [Question set 1](#question-set-1)
- [Question set 2](#question-set-2)

> **Q: what is page.tsx is it a vite project?**
>
> No — this is a **Next.js** project (App Router), not Vite. There's no `vite.config.ts`, and `package.json`'s scripts use `next dev`/`next build`/`next start`, confirmed by `next.config.ts` and `next` v16.1.1 in dependencies.
>
> **What `page.tsx` is:** In Next.js's App Router, `src/app/page.tsx` is a special filename the framework recognizes automatically — it defines the UI for a route based on its folder location. Since it lives directly in `src/app/`, it's the component rendered for the root route `/` (the homepage). If you later added `src/app/settings/page.tsx`, that would automatically become the page for `/settings` — no manual route configuration needed, unlike Vite (which is just a bundler/dev-server and leaves routing entirely up to you, e.g. via React Router).
>
> So the rough equivalence is:
> - **Vite** = a build tool/dev server, routing is DIY.
> - **Next.js** = a full framework where the file system *is* the router, and `page.tsx` is the reserved name for "this folder's route content." `layout.tsx` (which this project also has) is the matching reserved name for shared wrapper UI around a page.

> **Q: what is the role of page.tsx and layout.tsx?**
>
> In Next.js's App Router, these are two reserved filenames with distinct jobs:
>
> **`src/app/layout.tsx` — the shared shell.** Wraps every page under it and stays mounted across navigations (it doesn't re-render when you move between routes). In this project it's the *root* layout, so it wraps the entire app. It sets up `<html>`/`<body>`, fonts, and global metadata; wraps everything in `ClerkProvider` → `ThemeProvider` → `ConvexClientProvider` so auth, theme, and the Convex connection are available to every page; renders the header (Sign In/Sign Up buttons or `UserButton`) that appears on every route; and renders `{children}` — that's where the current page's content gets slotted in.
>
> **`src/app/page.tsx` — the actual route content.** Defines what's unique to one specific route. Since it sits directly in `src/app/`, it's the page for `/` — this project's homepage. It's *inside* `{children}` from the layout above it, and holds the stuff that's specific to this one screen: the "Create Project" form, the live projects list, the mode toggle — none of which appear on other routes, unlike the header in `layout.tsx`.
>
> **The relationship:** for any URL, Next.js renders every `layout.tsx` from the root down to that route, nested outside-in, with that route's `page.tsx` as the innermost `{children}`. Right now there's only one layout and one page, so it's simple: `layout.tsx` → `page.tsx`. If you added `src/app/settings/page.tsx`, it would automatically inherit this same root layout (header, providers, fonts) and only need to define its own unique content — that's the reuse layouts are for.

> **Q: what is the idea of wrapping Clerk->Theme->convex Is this a react thing or nextjs?**
>
> It's a **React pattern**, not Next.js-specific — Next.js just gives you `layout.tsx` as a canonical place to do it once, app-wide.
>
> **The underlying mechanism: React Context.** Each of `ClerkProvider`, `ThemeProvider`, and `ConvexClientProvider` (internally, `ConvexProviderWithClerk`) is a Context Provider — a component that makes some value (session state, theme, a Convex client) available to every descendant via `useContext`-based hooks (`useAuth`, `useTheme`, `useConvexAuth`, etc.), without threading props down manually through every level. This is plain React, and works identically in a Vite app, Create React App, or anywhere else.
>
> **Why this specific nesting order matters — and it's not arbitrary:**
> ```
> ClerkProvider
>   -> ThemeProvider
>     -> ConvexClientProvider   (internally: ConvexProviderWithClerk)
> ```
> `ConvexProviderWithClerk` (`src/app/ConvexClientProvider.tsx`) calls Clerk's `useAuth()` internally to read the session and attach a token to the Convex connection. `useAuth()` only works if a `ClerkProvider` exists somewhere *above* it in the tree — React context lookups walk upward, so a child can read a provider's value, but a provider can't read one nested inside it. So **Clerk must wrap Convex, not the other way around** — that's a hard dependency, not a style choice. `ThemeProvider` in the middle has no such dependency on Clerk or Convex (it only manages a CSS class via `next-themes`), so its position between the two is really just convention/readability here, not a requirement.
>
> **Where Next.js comes in:** it doesn't invent this pattern — it just gives you exactly one obvious place to set it up once for the whole app: the root `src/app/layout.tsx`, since every route renders inside it. In a plain Vite/React app you'd do the identical nesting by hand in your top-level `App.tsx` or `main.tsx` instead.

<a id="current-project-status"></a>

## <span style="color: #eab308">Current Project Status</span>

**Project:** `my_polaris`.

**What's actually built, functionally:**
- A single homepage (`src/app/page.tsx`) that renders:
  - A header with Clerk **Sign In** / **Sign Up** buttons when signed out, and a **UserButton** (avatar menu) when signed in — defined in `src/app/layout.tsx`.
  - A light/dark **mode toggle** (`src/components/mode-toggle.tsx`, powered by `next-themes`).
  - A **"Project name" input + "Create Project" button**, wired to a real Convex mutation.
  - A **live list** of the signed-in user's projects, wired to a real Convex query — updates automatically, no refresh needed.
- A Convex backend (`convex/`) with one table, `projects` (fields: `name`, `ownerId`, `importStatus`), and two functions in `convex/projects.ts`: `create1` (insert a project) and `get1` (list the current user's projects).
- Convex is wired to Clerk for authentication (`convex/auth.config.ts` + `ConvexProviderWithClerk` in `src/app/ConvexClientProvider.tsx`), so every Convex call knows *who* is calling and can be scoped to that user.
- Around 60 shadcn/ui components are already generated under `src/components/ui/` (button, input, dialog, sidebar, calendar, chart, form, etc.). Most are **not wired into the page yet** — they're scaffolding the `shadcn` CLI generated for future use.

**In short:** this is an early-stage scaffold — working auth, a working database, and one working create/list flow — not yet a full multi-page app.

<a id="tech-stack-and-architecture"></a>

## <span style="color: #eab308">Tech Stack and Architecture</span>

> **Core stack**
> | Layer | Tool | Role |
> |---|---|---|
> | Framework | Next.js 16.1.1 (App Router, Turbopack) | Routing, rendering, dev server |
> | UI runtime | React 19.2.3 / react-dom 19.2.3 | Component rendering |
> | Styling | Tailwind CSS v4 (+ `tailwind-merge`, `tw-animate-css`) | Utility-class styling |
> | Component library | shadcn/ui on Radix primitives (`radix-ui`, `@base-ui/react`) | Pre-built accessible UI components in `src/components/ui/` |
> | Auth | Clerk (`@clerk/nextjs`) | Sign in/up, sessions, `src/proxy.ts` middleware |
> | Backend/data | Convex (`convex`) | Realtime database + serverless functions, synced via `npx convex dev` |
> | Auth <-> Convex bridge | `convex/react-clerk`'s `ConvexProviderWithClerk` | Attaches the Clerk session token to every Convex request |
> | Forms (installed, not yet used) | `react-hook-form` + `zod` + `@hookform/resolvers` | Form state + validation, ready for future forms |
> | Icons | `lucide-react` | Icon set used by shadcn components |
> | Theming | `next-themes` | System/light/dark mode |
> | Language | TypeScript 5 | Type safety across app + Convex functions |

**Request/render flow:**

Browser → Next.js dev server → `src/app/layout.tsx` renders providers in this order:

```
ClerkProvider                 (Clerk session available everywhere)
  -> ThemeProvider             (next-themes: applies light/dark class)
    -> ConvexClientProvider    (opens the Convex WebSocket, attaches Clerk token)
      -> page content (src/app/page.tsx)
```

The Convex client talks **directly** to your Convex cloud deployment over its own WebSocket — it is not proxied through the Next.js server. `src/proxy.ts` (Clerk's middleware, renamed from `middleware.ts` in Next.js 16) runs server-side on every matching request to attach auth context, but it does not block any routes in this project yet.

<a id="environment-variables-you-need"></a>

## <span style="color: #eab308">Environment Variables You Need</span>

Two files hold them, and neither is meant to be shared:

- **`.env`** — Clerk keys + Convex keys. Loaded by both the Next.js server and, where prefixed `NEXT_PUBLIC_`, the browser.
- **`.env.local`** — duplicates the three Convex vars (a leftover from before they were moved into `.env`; harmless as long as the values agree, since Next.js merges both files).

> **Variables (names only — values are secrets, not shown)**
> | Variable | Used in | Purpose |
> |---|---|---|
> | `NEXT_PUBLIC_CLERK_PUBLISHABLE_KEY` | `ClerkProvider` (`src/app/layout.tsx`) | Clerk's public/browser-safe key |
> | `CLERK_SECRET_KEY` | `clerkMiddleware()` (`src/proxy.ts`) | Clerk's server-side secret |
> | `CLERK_JWT_ISSUER_DOMAIN` | `convex/auth.config.ts` | Lets Convex verify Clerk's session JWTs |
> | `CONVEX_DEPLOYMENT` | read by the `convex` CLI | Tells `npx convex dev` / `npx convex deploy` which deployment to sync |
> | `NEXT_PUBLIC_CONVEX_URL` | `ConvexClientProvider.tsx` | The browser-facing Convex deployment URL |
> | `NEXT_PUBLIC_CONVEX_SITE_URL` | generated by `convex dev` | Convex's HTTP Actions site URL (not used by any code yet) |

If you ever copy this project to a new machine or hand it to someone else, they will need their own copies of these two files — that's expected; these files simply live on disk, local to each machine.

<a id="how-the-app-starts-step-by-step"></a>

## <span style="color: #eab308">How the App Starts, Step by Step</span>

**1. `npx convex dev` connects to your Convex deployment**
Reads `CONVEX_DEPLOYMENT`, pushes `convex/schema.ts` and every function in `convex/*.ts` to that deployment, and — while left running — keeps `convex/_generated/` (`api.d.ts`, `api.js`, `server.d.ts`, …) in sync. This generated folder is what makes `api.projects.create1` and `api.projects.get1` exist as typed references in `src/app/page.tsx`. Nothing Convex-related works without this running.

**2. `npm run dev` starts the Next.js dev server**
Turbopack builds and serves the app on `http://localhost:3000`.

**3. Browser requests `/` → `src/app/layout.tsx` renders**
`ClerkProvider` wraps everything first, reading `NEXT_PUBLIC_CLERK_PUBLISHABLE_KEY` so Clerk's session state (signed in / signed out) is available app-wide.

**4. `ThemeProvider` (next-themes) applies the theme class**
Before paint, so the correct light/dark styling shows immediately (`suppressHydrationWarning` on `<html>` avoids a server/client mismatch warning here).

**5. `ConvexClientProvider` opens the Convex connection**
Creates one `ConvexReactClient` pointed at `NEXT_PUBLIC_CONVEX_URL`, wrapped in `ConvexProviderWithClerk`, which calls Clerk's `useAuth` to fetch the current session token and attaches it to the Convex WebSocket connection.

**6. `src/proxy.ts`'s `clerkMiddleware()` runs server-side**
On every matching request (per its `config.matcher`), attaching auth context so server components / route handlers *could* call `auth()` or `currentUser()` — nothing in this project does yet, but the wiring is in place.

**7. `Home` (`src/app/page.tsx`) mounts in the browser**
`useConvexAuth()` reports whether Clerk **and** Convex both agree you're signed in. While `isAuthenticated` is `false`, the `get1` query is skipped (passed `"skip"` instead of `{}`). Once `true`, it subscribes for real, and your projects list appears — updating live on every future create, with no manual refresh.

<a id="what-to-type-in-the-terminal"></a>

## <span style="color: #eab308">What to Type in the Terminal</span>

You need **two terminals running at the same time** — one for Convex, one for Next.js. Both stay open while you work; close either with `Ctrl+C` when you're done for the day.

**Terminal A — Convex (leave running):**
```bash
cd my_polaris
npx convex dev
```
First time ever on a machine, this will prompt you to log in to Convex and confirm the project link; after that it just syncs silently in the background.

**Terminal B — Next.js (leave running):**
```bash
cd my_polaris
npm run dev
```

Then open **http://localhost:3000** in your browser.

**If `node_modules` is missing, or after pulling changes that touch `package.json`:**
```bash
npm install
```
Run this once before `npm run dev`.

> **Quick reference**
> | Command | What it does |
> |---|---|
> | `npm run dev` | Start the Next.js dev server (Turbopack, hot reload) |
> | `npx convex dev` | Sync `convex/` functions + schema, watch for changes |
> | `npm run build` | Production build |
> | `npm run start` | Run the production build locally |
> | `npm run lint` | Run ESLint |

<a id="troubleshooting-and-notes"></a>

## <span style="color: #eab308">Troubleshooting and Notes</span>

**Sign In / Sign Up buttons don't render, or a Clerk error appears**
Check `.env`'s `NEXT_PUBLIC_CLERK_PUBLISHABLE_KEY` and `CLERK_SECRET_KEY` are set, and restart `npm run dev` — environment variables are only read when the dev server starts, so editing `.env` while it's running has no effect until restart.

**"Create Project" throws `Not authenticated`**
You're not signed in. `ctx.auth.getUserIdentity()` in `convex/projects.ts` requires an active Clerk session — sign in first via the header button.

**Convex calls fail entirely, or `api.projects...` shows as undefined in the editor**
Make sure `npx convex dev` is actually running in its own terminal — it's what keeps `convex/_generated/` and the live schema in sync. The Next.js dev server does **not** start it for you; the two are independent processes.

**`npm error ENOENT ... Could not read package.json`**
This happened once before, in an earlier session, because `npm` was run from the *parent* folder (the old `polaris/`, one level above where `package.json` actually lives) instead of from inside `my_polaris/`. Always run `npm`/`npx` commands from inside the folder that directly contains `package.json`.

### One subtlety
`.env` and `.env.local` both define the same three Convex variables. Next.js loads `.env.local` with higher precedence than `.env`, so `.env.local`'s copies are the ones actually in effect — the `.env` copies are redundant but harmless as long as both stay in sync. If they ever drift apart, `.env.local` wins.

<a id="question-set-1"></a>

## <span style="color: #eab308">Question set 1</span>

> **Q: explain the Input and its handlers. When Enter is pressed, then handleCreate then a project is created in convex. so what is onChange doing?**
>
> Line by line:
> - **`<Input placeholder="Project name" ...>`** — the actual text field.
> - **`value={newProjectName}`** — this makes it a **controlled component**: whatever the box displays is dictated entirely by React state (`newProjectName`, from `useState("")`), not by the browser's own internal input state.
> - **`onChange={(e) => setNewProjectName(e.target.value)}`** — fires on **every single keystroke** (each character typed or deleted), not on Enter.
> - **`onKeyDown={(e) => { if (e.key === "Enter") handleCreate(); }}`** — fires on every key press too, but only *acts* when that key is `"Enter"`.
>
> **What `onChange` is doing, specifically:** it has nothing to do with Convex or creating the project — it's purely local UI bookkeeping. Because `value={newProjectName}` pins the box's displayed text to React state, the box can *only* show new characters if something updates that state in response to typing. `onChange` is that something: each keystroke calls `setNewProjectName(e.target.value)`, which re-renders the component with the box's `value` now equal to whatever you just typed.
>
> If you deleted `onChange`, the input would appear **frozen** — you could click into it and type, but since `value` never changes (nothing updates `newProjectName`), React would keep snapping the displayed text back to the old value on every render. This is the classic "controlled input needs `value` + `onChange` together" pairing.
>
> **So the full sequence when you type "My App" and hit Enter:**
> 1. Each keystroke (`M`, `y`, ` `, `A`, `p`, `p`) individually triggers `onChange` → `newProjectName` accumulates to `"My App"` in state, purely client-side, no network calls.
> 2. Pressing Enter triggers `onKeyDown` → `handleCreate()` runs, which reads the *already-typed* `newProjectName` from state, trims it, and sends it to Convex via `createProject({ name })`.
>
> So `onChange` is what lets you see what you're typing at all; `onKeyDown`/`handleCreate` is what happens *after*, using the value `onChange` already built up.

> **Q: after I press Enter, handleCreate is creating a new project in convex, but how the name that was typed is used by handleCreate?**
>
> Via **closure**, not an argument.
> ```tsx
> const [newProjectName, setNewProjectName] = useState("");
>
> const handleCreate = async () => {
>   const name = newProjectName.trim();
>   ...
>   await createProject({ name });
> };
> ```
> `handleCreate` takes no parameters. `Home` is a function component, so it re-runs top-to-bottom on every state change, and a **new** `handleCreate` function is created each render — that new function closes over (captures) whatever `newProjectName` equals *in that render*. Every keystroke's `onChange` (`setNewProjectName(e.target.value)`) triggers a re-render, producing a fresh `handleCreate` bound to the just-typed value, which gets wired up again to the `<Input>`'s `onKeyDown` and the `<Button>`'s `onClick`. So when you press Enter, the `handleCreate` that actually runs is whichever one is currently attached — the one from the most recent render — and it reads `newProjectName` straight out of that closure, not as an argument passed in.

> **Q: why is it `handleCreate` (no parens) on the button's `onClick`, but `handleCreate()` (with parens) inside the `onKeyDown` handler? If the `onKeyDown` handler already creates the project, what does the button do?**
>
> **Summary:** both trigger the exact same `handleCreate`, with the exact same result — pressing Enter and clicking the button are genuinely equivalent. The parens difference is just the two forms described below: the button passes a bare reference (`{handleCreate}`), while `onKeyDown` has to use an inline wrapper (`{(e) => { if (e.key === "Enter") handleCreate(); }}`) because it needs to filter out every non-Enter key before deciding to call it — a bare reference there would fire on every single keystroke instead.
>
> Given both paths lead to the same place, the button still earns its spot for three reasons: **discoverability** (not everyone expects Enter to submit), **touch devices** (no reliable Enter-equivalent there), and **visible feedback** — the button visibly grays out via `disabled={!newProjectName.trim()}` when the input is empty, while the Enter-key path just silently does nothing in that case (`handleCreate`'s own guard exits quietly, with no visual cue).
>
> As for how a bare reference like `{handleCreate}` ever actually gets called, and why render timing isn't what triggers it — see the syntax breakdown below, which explains the general mechanism (bare reference vs. inline wrapper) that this specific case is just one example of.

> **Q: my question is about how each handler is wired up in the code — is it a syntax issue or a pattern issue? What is the syntax of `onClick={handleCreate}` compared to the `onKeyDown` handler? Could I write `onClick` as an arrow function that invokes `handleCreate()` immediately instead?**
>
> It's a **pattern choice, not a syntax requirement.** JSX doesn't have two different kinds of event-prop syntax; `onClick` and `onKeyDown` both follow the exact same rule.
>
> **The one syntax rule:** any JSX attribute is written `propName={expression}` — the `{}` just embeds a JavaScript expression, and whatever that expression evaluates to becomes the prop's value. `value={newProjectName}` and `onClick={handleCreate}` follow the identical rule — one just happens to hold a string, the other happens to hold a function.
>
> Given that one rule, there are two kinds of expression you can put inside `{}`, and both are equally legal on `onClick` **or** `onKeyDown`:
> 1. **A bare identifier** — `{handleCreate}` — evaluates to the function object itself. React stores that reference and calls it later, passing the event as the argument.
> 2. **An inline arrow function** — `{() => handleCreate()}` or `{(e) => { ... }}` — evaluates to a *new* function, created fresh each render, whose body you fully control. React calls *this* one on the event, and your code decides what happens next.
>
> **So yes — this would work, and behave identically to what's already there:**
> ```tsx
> <Button onClick={() => handleCreate()} disabled={!newProjectName.trim()}>
> ```
> The wrapper receives the click event as an implicit argument, ignores it, and calls `handleCreate()` explicitly. Same outcome either way.
>
> **So why is one written bare and the other wrapped, if both forms work everywhere?** Purely because of what each one *needs* to do before deciding to call `handleCreate`:
> - The button's click needs **no filtering** — a click is always meant to trigger creation. The bare reference is the simplest sufficient form; wrapping it would work but is unnecessary extra code.
> - `onKeyDown` fires on *every* key, so it **must** run some logic first (`if (e.key === "Enter")`) to decide *whether* to call `handleCreate` at all. That logic has to live somewhere, so an inline arrow function is the only way to fit it in.
>
> **One related gotcha worth knowing:** the bare-reference form only works safely when the function either takes no parameters (like `handleCreate`) or expects the event as its first argument — because React always calls it with the event. If a handler needed some *other* value — say, a function like `handleDelete(projectId)` — you'd be forced into the wrapper form: `onClick={() => handleDelete(project._id)}`, never `onClick={handleDelete}`, because the bare form would incorrectly hand it the click event instead of the id.

<a id="question-set-2"></a>

## <span style="color: #eab308">Question set 2</span>

> **Q: explain `const projects = useQuery(api.projects.get1, isAuthenticated ? {} : "skip");` and the `projects?.map(({ _id, ownerId, name }) => (...))` block that renders each project.**
>
> **`const projects = useQuery(api.projects.get1, isAuthenticated ? {} : "skip");`**
> - **`useQuery`** (from `convex/react`) doesn't work like a normal async function call — it registers a **live, standing subscription** to a Convex query. Convex re-runs `get1` on the server and pushes fresh results down a WebSocket automatically whenever the underlying data changes, no manual refetching or polling involved.
> - **`api.projects.get1`** is a typed reference (generated into `convex/_generated/api`) pointing at the `get1` query defined in `convex/projects.ts` — the one that checks `ctx.auth.getUserIdentity()` and returns only the current user's own projects.
> - **`isAuthenticated ? {} : "skip"`** is the second argument — the query's *arguments*, or the sentinel `"skip"`. `isAuthenticated` comes from `useConvexAuth()`. This is a ternary expression: while `isAuthenticated` is `false`, the value is `"skip"` — a special string `useQuery` understands as "don't run this query at all." `projects` stays `undefined` the entire time. Once Convex confirms a signed-in identity, `isAuthenticated` flips to `true`, the argument becomes `{}` (an empty args object, since `get1` takes no parameters), and `useQuery` starts the real subscription — `get1` runs, filtered to `ownerId === identity.subject`, and `projects` becomes the resulting array.
> - Any time that second-argument *value* changes (`"skip"` ↔ `{}`), Convex tears down the old subscription (if any) and starts fresh — this is what makes sign-in/sign-out correctly start/stop the live data feed.
>
> **`{projects?.map(({ _id, ownerId, name }) => ( ... ))}`**
> - **`projects?.`** — optional chaining. Since `projects` is `undefined` until the query above actually returns data (or is skipped entirely while signed out), this guards against calling `.map()` on `undefined`, which would throw. If `projects` is `undefined`, the whole expression evaluates to `undefined`, and React simply renders nothing there.
> - **`.map(({ _id, ownerId, name }) => ( ... ))`** — once `projects` is a real array, this transforms each project *document* into a piece of JSX. `({ _id, ownerId, name })` destructures those three fields directly out of each element as it's passed in (a Convex document also carries `_creationTime`, unused here). The arrow's body is wrapped in `( ... )` right after `=>` — the implicit-return form, so whatever JSX is inside is automatically that array element's produced value.
> - **`<div className="..." key={_id.toString()}>`** — one styled box per project. `key` is required by React whenever rendering an array of elements, so it can correctly track which box corresponds to which project across re-renders (added/removed/reordered items). `_id` is the right choice here since it's guaranteed unique per document — unlike `ownerId`, which would be identical across every project this particular query returns (they all belong to the same signed-in user).
> - **`{name} ({ownerId})`** — renders the project's name, followed by its owner's Clerk user ID in parentheses, as the box's visible text.
>
> **How the two connect:** `useQuery` is what keeps `projects` continuously up to date in the background (or `undefined` while signed out); `.map()` is what turns whatever `projects` currently holds into visible boxes on the page, re-running automatically every time React re-renders in response to that live data changing.

> **Q: would it make sense to create `providers.tsx` and put everything under `<Authenticated>`, as in the reference project's `polaris-main/src/components/providers.tsx`?**
>
> Yes, it would — that's a cleaner pattern than what's currently in `page.tsx`. Right now, auth-gating is done ad hoc per-query (`isAuthenticated ? {} : "skip"` plus manual empty-state handling); the reference's `providers.tsx` moves that decision to one place at the root, using Convex's `<Authenticated>`/`<Unauthenticated>`/`<AuthLoading>` components to show the real app, a sign-in prompt, or a loading spinner respectively — so individual pages/queries can just assume they're authenticated and skip the boilerplate entirely.
>
> The main tradeoff: this makes the **entire app** require sign-in to see anything at all (no page renders unless you're logged in) — which fits a private tool like the reference app, but is a real behavior change from `my_polaris` today, where the page is currently visible/usable-looking even when signed out (only the actual Convex calls are blocked). If signed-out visitors should ever see any public-facing content, `<Authenticated>` would need to wrap only part of the tree, not everything.

> **Q: why does `layout.tsx` not make sure that if anything in `{children}` is not authenticated, then it would not be shown?**
>
> Because nothing in `layout.tsx` currently tells it to gate `{children}` — the auth-conditional rendering that exists there (`<Show when="signed-out">` / `<Show when="signed-in">`) is only wrapped around the header's Sign In/Sign Up buttons vs. the `UserButton`. `{children}` right after the header is rendered **unconditionally** — there's no `<Show>`, no `<Authenticated>`, no check of any kind around it. It's simply never been written; not something the code is attempting and failing at, just absent.
>
> **Where auth enforcement actually happens instead, in this app: the data layer, not the UI layer.**
> - In `page.tsx`, `useQuery(api.projects.get1, isAuthenticated ? {} : "skip")` means the query is *skipped* when signed out — so `projects` stays `undefined` and nothing renders in that specific list.
> - In `convex/projects.ts`, both `create1` and `get1` call `ctx.auth.getUserIdentity()` and throw if there's no identity — so even if someone bypassed the UI, the actual database operations reject unauthenticated callers.
>
> So the **data** is protected, but the **UI shell** is not — a signed-out visitor still sees the full page structure: the "Project name" input, the "Create Project" button, the whole layout. They just can't successfully create anything (the mutation throws) or see any projects (the query never runs). It's a page that *looks* interactive but silently does nothing useful when you're not signed in, rather than a page that's replaced by a sign-in prompt.
>
> This is exactly the gap the reference project's `providers.tsx` pattern closes: wrapping `{children}` in Convex's `<Authenticated>` would stop that content from rendering *at all* when signed out, showing an `<Unauthenticated>` view (like a "please sign in" screen) instead — moving the enforcement from "the data quietly refuses to load" to "the UI itself refuses to appear."

### Plan: landing page when signed out, current app when signed in

This is the `<Authenticated>` / `<Unauthenticated>` pattern from the reference project, applied to a concrete goal: signed-out visitors see only a title and sign-in/sign-up controls (the `WelcomeView`); signed-in users see exactly what the app already shows today (the project-name input, the "Create Project" button, and the live project list).

**1. Create the `WelcomeView` component** — a new file, e.g. `src/components/welcome-view.tsx`:
```tsx
"use client";

import { SignInButton, SignUpButton } from "@clerk/nextjs";
import { Button } from "@/components/ui/button";

export function WelcomeView() {
  return (
    <div className="flex min-h-screen flex-col items-center justify-center gap-6">
      <h1 className="text-3xl font-bold">Welcome to My_Polaris</h1>
      <div className="flex gap-3">
        <SignInButton>
          <Button variant="outline">Sign In</Button>
        </SignInButton>
        <SignUpButton>
          <Button>Sign Up</Button>
        </SignUpButton>
      </div>
    </div>
  );
}
```
This mirrors the shape of the reference project's `UnauthenticatedView`, but simplified to just a title plus the two Clerk buttons, styled full-height and centered so it reads as an actual landing page rather than a small header corner.

**2. Wrap `{children}` in `layout.tsx` with `<Authenticated>` / `<Unauthenticated>`, replacing the current `<Show>`-based header logic:**
```tsx
import { Authenticated, Unauthenticated } from "convex/react";
import { WelcomeView } from "@/components/welcome-view";

...

<ConvexClientProvider>
  <Authenticated>
    {children}
  </Authenticated>
  <Unauthenticated>
    <WelcomeView />
  </Unauthenticated>
</ConvexClientProvider>
```
This replaces the current header entirely for signed-out users — `WelcomeView` becomes the whole page they see, not a header bolted onto existing content. The existing `UserButton` (for signed-in users) would move into whatever header `{children}`'s own page renders, since `{children}` is now only reached once authenticated.

**3. Leave `page.tsx` almost untouched.** Since `{children}` (which is `page.tsx`'s output) now only renders once `<Authenticated>` has confirmed a session, the `isAuthenticated ? {} : "skip"` guard on `useQuery` becomes redundant — `get1` can be called unconditionally, since by the time this component ever mounts, Convex already agrees the user is signed in. It's safe to leave the guard in place too (harmless belt-and-suspenders), so this step is optional cleanup, not a requirement.

**4. Result:** signed out → `<Unauthenticated>` renders `WelcomeView` (title + sign-in/sign-up), and nothing else in the app is reachable or rendered. Signed in → `<Authenticated>` renders `{children}`, i.e. today's project-creation-and-list page, unchanged in behavior. This is the same underlying mechanism as `providers.tsx` in the reference project, just applied directly inside the existing `layout.tsx` rather than extracted into a separate file — extracting it into its own `providers.tsx` (as discussed above) is a separate, independent decision from adding the landing page itself.